In [ ]:
import os
import glob
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs


folder_path = '/home/users/yao.wei/Desktop/cyp_toxicogenomics/zinc/SMILES'
smi_files = glob.glob(os.path.join(folder_path, '*.smi'))

# List of substrate/toxicant SMILES
substrate_smiles_list = [
    "C1=CC=C2C(=C1)C(=NN=C2NN)NN",        # Dihydralazine_s1
    "COC1=C2C3=C(C(=O)CC3)C(=O)OC2=C4[C@@H]5C=CO[C@@H]5OC4=C1", # Aflatoxin B1_s2
    "CCC1=CC=C(C=C1)C(=O)C2=CC(=C(C(=C2)O)O)[N+](=O)[O-]",    # Tolcapone_s3
    "CC(=O)NC1=CC=C(C=C1)O",  # Acetaminophen_s4
    "CCOC1=CC=C(C=C1)NC(=O)C",  #Phenacetin_s5
    "C1[C@@H]2[C@@H](C2N)CN1C3=C(C=C4C(=O)C(=CN(C4=N3)C5=C(C=C(C=C5)F)F)C(=O)O)F",  # Trovafloacin_s6
    "CC(=O)NC1=CC2=C(C=C1)C3=CC=CC=C3C2",   # 2-Acetylaminofluorene_s7
    "C1=CC=C(C=C1)C2=CC=C(C=C2)N",  # 4-Aminobiphenyl_s8
    "C1=CC=C2C3=C4C(=CC2=C1)C=CC5=C4C(=CC=C5)C=C3",   # Benzo[a]pyrene (BaP)_s9
    "CN1C2=CC(=C3C(=C2N=C1N)C=CC=N3)O", # 2-amino-3-methylimidazo[4,5-f]quinoline (IQ)_s10
    "CC(C)(C1=CC=C(C=C1)O)C2=CC=C(C=C2)O",  # bisphenol A_s11
    "C1C[C@H](N(C1)N=O)C2=CN=CC=C2"  # N'-nitrosonornicotine_s12
]

# Convert each substrate SMILES to a fingerprint
toxic_fps = []
for smi in substrate_smiles_list:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
        toxic_fps.append(fp)

# Load SMILES and ZINC IDs from all files
db_entries = []  # List of (smiles, zinc_id)
for file in smi_files:
    with open(file, 'r') as f:
        next(f)  # Skip header
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                smi, zinc_id = parts[0], parts[1]
                db_entries.append((smi, zinc_id))

# Filter valid molecules and compute fingerprints
db_mols = []
valid_entries = []
for smi, zinc_id in db_entries:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        db_mols.append(mol)
        valid_entries.append((smi, zinc_id))

db_fps = [AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048) for mol in db_mols]

# Compute max similarity to any known toxicant
threshold = 0.5
toxic_like = []
for (smi, zinc_id), db_fp in zip(valid_entries, db_fps):
    max_sim = max(DataStructs.TanimotoSimilarity(db_fp, tox_fp) for tox_fp in toxic_fps)
    if max_sim >= threshold:
        toxic_like.append((smi, zinc_id, max_sim))


output_file = '/home/users/yao.wei/Desktop/cyp_toxicogenomics/zinc/substrate_toxic_dataset.txt'
with open(output_file, 'w') as out_f:
    out_f.write("SMILES\tZINC_ID\tSimilarity\n")
    for smi, zinc_id, sim in toxic_like:
        out_f.write(f"{smi}\t{zinc_id}\t{sim:.3f}\n")

print(f"Found {len(toxic_like)} substrate toxic-like molecules. Saved to {output_file}")


[22:39:14] DEPRECATION WARNING: please use MorganGenerator
[22:39:14] DEPRECATION WARNING: please use MorganGenerator
[22:39:14] DEPRECATION WARNING: please use MorganGenerator
[22:39:14] DEPRECATION WARNING: please use MorganGenerator
[22:39:14] DEPRECATION WARNING: please use MorganGenerator
[22:39:14] DEPRECATION WARNING: please use MorganGenerator
[22:39:14] DEPRECATION WARNING: please use MorganGenerator
[22:39:14] DEPRECATION WARNING: please use MorganGenerator
[22:39:14] DEPRECATION WARNING: please use MorganGenerator
[22:39:14] DEPRECATION WARNING: please use MorganGenerator
[22:39:14] DEPRECATION WARNING: please use MorganGenerator
